In [1]:
# This is our third and last model implementation (siamense model)
# This is an deep learning model which is specifically involved to perform the approach 
# of similar species indetification by using metrix learning 
# Individual animal re-identification can not perform because CCT20 datasets does not 
# provide consistent animal identities
# so we need to perform this using species level similarity verification using triplet loss
# We are goinng to use 3 images :--
# 1. Reference image , 2. positive image , 3. negative image 
# Core objectives :-- bring embeddings off same species together and push different species embeddings apart 
# Same 80/20 splits is going to perform earlier used in ResNet50 and depth fusion model



In [2]:
import torch
import os
import json
from torch.utils.data import Dataset, Subset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np
import random

processing_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {processing_device}")

Using device: cuda


In [3]:
transform_image = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

standard_deviation = [0.229, 0.224, 0.225, 0.225]
Mean = [0.485, 0.456, 0.406, 0.5]
transform_fusion = transforms.Compose([
    transforms.Normalize(Mean, standard_deviation)
])

In [4]:
class Camera_Dataset(Dataset):
    def __init__(self, images_folder, path_annotations, transform=None):
        self.images_folder = images_folder
        self.transform = transform
        with open(path_annotations, "r", encoding="utf-8") as f:
            self.annotations = json.load(f)
        final_ids = sorted(set(a["category_id"] for a in self.annotations))
        self.label_map = {cat_id: slot for slot, cat_id in enumerate(final_ids)}

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        annotation_data = self.annotations[index]
        img_path = os.path.join(self.images_folder, annotation_data["image_id"] + ".jpg")
        Img = Image.open(img_path).convert("RGB")
        lb = self.label_map[annotation_data["category_id"]]
        if self.transform:
            Img = self.transform(Img)
        return Img, lb

d = Camera_Dataset(
    r"D:\Camera_trap\data\processed_data",
    r"D:\Camera_trap\data\processed_data\prepared_annotations.json",
    transform=transform_image
)
print(f"d size = {len(d)}")

d size = 44232


In [5]:
torch.manual_seed(42)
dataset_train_size = int(len(d) * 0.8)
dataset_test_size = len(d) - dataset_train_size
dataset_train, dataset_test = random_split(d, [dataset_train_size, dataset_test_size])
print(f"Train size = {len(dataset_train)}")
print(f"Test size = {len(dataset_test)}")

Train size = 35385
Test size = 8847


In [6]:
class Camera_Triplet_Dataset(Dataset):
    def __init__(self, images_folder, depth_npy_folder, path_annotations, transform=None):
        self.images_folder = images_folder
        self.depth_npy_folder = depth_npy_folder
        self.transform = transform

        with open(path_annotations, "r", encoding="utf-8") as f:
            self.annotations = json.load(f)

        final_ids = sorted(set(a["category_id"] for a in self.annotations))
        self.label_map = {cat_id: slot for slot, cat_id in enumerate(final_ids)}

        self.by_class = {}
        for idx, ann in enumerate(self.annotations):
            label = self.label_map[ann["category_id"]]
            self.by_class.setdefault(label, []).append(idx)

    def __len__(self):
        return len(self.annotations)

    def _load_fused_image(self, index):
        annotation_data = self.annotations[index]
        image_id = annotation_data["image_id"]

        rgb_img = Image.open(os.path.join(self.images_folder, image_id + ".jpg")).convert("RGB")
        rgb_img = rgb_img.resize((224, 224))
        rgb_array = np.array(rgb_img).astype(np.float32) / 255.0

        depth_array = np.load(os.path.join(self.depth_npy_folder, image_id + ".npy")).astype(np.float32)
        depth_img = Image.fromarray(depth_array.astype(np.uint8)).resize((224, 224))
        depth_array = np.array(depth_img).astype(np.float32) / 255.0

        depth_array = np.expand_dims(depth_array, axis=-1)
        fused = np.concatenate([rgb_array, depth_array], axis=-1)
        fused_tensor = torch.from_numpy(fused).permute(2, 0, 1).float()

        if self.transform:
            fused_tensor = self.transform(fused_tensor)

        return fused_tensor

    def __getitem__(self, index):
        anchor_label = self.label_map[self.annotations[index]["category_id"]]

        positive_idx = index
        while positive_idx == index:
            positive_idx = random.choice(self.by_class[anchor_label])

        negative_label = anchor_label
        while negative_label == anchor_label:
            negative_label = random.choice(list(self.by_class.keys()))
        negative_idx = random.choice(self.by_class[negative_label])

        anchor = self._load_fused_image(index)
        positive = self._load_fused_image(positive_idx)
        negative = self._load_fused_image(negative_idx)

        return anchor, positive, negative

In [7]:
d_triplet = Camera_Triplet_Dataset(
    r"D:\Camera_trap\data\processed_data",
    r"D:\Camera_trap\data\processed_data\depth_maps\npy",
    r"D:\Camera_trap\data\processed_data\prepared_annotations.json",
    transform=transform_fusion
)
print(f"Triplet dataset size = {len(d_triplet)}")

triplet_train = Subset(d_triplet, dataset_train.indices)
triplet_test = Subset(d_triplet, dataset_test.indices)

triplet_train_loader = DataLoader(triplet_train, batch_size=16, shuffle=True)
triplet_test_loader = DataLoader(triplet_test, batch_size=16, shuffle=False)

anchor, positive, negative = next(iter(triplet_train_loader))
print("Anchor batch shape:", anchor.shape)

Triplet dataset size = 44232
Anchor batch shape: torch.Size([16, 4, 224, 224])


In [8]:
import torchvision.models as models
import torch.nn as nn
import torch.nn.functional as F

def get_embedding_resnet50(embedding_dim=128):
    model = models.resnet50(weights="IMAGENET1K_V1")

    # Modify first conv layer for 4-channel input (RGB + Depth), same as fusion model
    old_conv = model.conv1
    new_conv = nn.Conv2d(4, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                          stride=old_conv.stride, padding=old_conv.padding, bias=False)
    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = old_conv.weight
        new_conv.weight[:, 3:, :, :] = old_conv.weight[:, :1, :, :]
    model.conv1 = new_conv

    # Replace the final classification layer with an embedding layer
    # instead of 13 class scores, output a small feature vector
    model.fc = nn.Linear(model.fc.in_features, embedding_dim)
    return model

class SiameseNetwork(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.backbone = get_embedding_resnet50(embedding_dim)

    def forward_one(self, x):
        embedding = self.backbone(x)
        # L2-normalize so embeddings live on a unit sphere — makes
        # distance comparisons stable and is standard practice for triplet loss
        embedding = F.normalize(embedding, p=2, dim=1)
        return embedding

    def forward(self, anchor, positive, negative):
        anchor_emb = self.forward_one(anchor)
        positive_emb = self.forward_one(positive)
        negative_emb = self.forward_one(negative)
        return anchor_emb, positive_emb, negative_emb

siamese_model = SiameseNetwork(embedding_dim=128).to(processing_device)
print("Siamese model ready")

Siamese model ready


In [ ]:
import torch.optim as optim

triplet_loss_fn = nn.TripletMarginLoss(margin=1.0, p=2)
siamese_optimizer = optim.Adam(siamese_model.parameters(), lr=0.0001)

siamese_epochs = 5

for epoch in range(siamese_epochs):
    siamese_model.train()
    current_loss = 0.00
    batch_count = 0

    for anchor, positive, negative in triplet_train_loader:
        anchor = anchor.to(processing_device)
        positive = positive.to(processing_device)
        negative = negative.to(processing_device)

        siamese_optimizer.zero_grad()

        anchor_emb, positive_emb, negative_emb = siamese_model(anchor, positive, negative)
        loss = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)

        loss.backward()
        siamese_optimizer.step()

        current_loss = current_loss + loss.item()
        batch_count += 1

        if batch_count % 50 == 0:
            print(f"  batch {batch_count}/{len(triplet_train_loader)} - running loss: {current_loss/batch_count:.4f}")

    current_epoch = epoch + 1
    average_loss = current_loss / len(triplet_train_loader)
    print(f"Epoch [{current_epoch}/{siamese_epochs}]")
    print(f"Average triplet loss: {average_loss:.4f}")
    print("Epoch complete")

torch.save(siamese_model.state_dict(), r"D:\Camera_trap\models\siamese_model_trained.pth")
print("Siamese model saved")

  batch 100/2212 - running loss: 0.6415


In [ ]:
siamese_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for anchor, positive, negative in triplet_test_loader:
        anchor = anchor.to(processing_device)
        positive = positive.to(processing_device)
        negative = negative.to(processing_device)

        anchor_emb, positive_emb, negative_emb = siamese_model(anchor, positive, negative)

        pos_distance = torch.norm(anchor_emb - positive_emb, p=2, dim=1)
        neg_distance = torch.norm(anchor_emb - negative_emb, p=2, dim=1)

        # Correct if positive pair is closer than negative pair
        correct += (pos_distance < neg_distance).sum().item()
        total += anchor.size(0)

verification_accuracy = (correct / total) * 100
print(f"Verification accuracy: {verification_accuracy:.2f}%")
print(f"({correct} / {total} triplets correctly ordered)")

In [2]:
siamese_model.eval()

all_pos_distances = []
all_neg_distances = []

with torch.no_grad():
    for anchor, positive, negative in triplet_test_loader:
        anchor = anchor.to(processing_device)
        positive = positive.to(processing_device)
        negative = negative.to(processing_device)

        anchor_emb, positive_emb, negative_emb = siamese_model(anchor, positive, negative)

        pos_distance = torch.norm(anchor_emb - positive_emb, p=2, dim=1)
        neg_distance = torch.norm(anchor_emb - negative_emb, p=2, dim=1)

        all_pos_distances.extend(pos_distance.cpu().numpy())
        all_neg_distances.extend(neg_distance.cpu().numpy())

import numpy as np

pos_mean = np.mean(all_pos_distances)
neg_mean = np.mean(all_neg_distances)

print(f"Average distance for SAME-species pairs (positive): {pos_mean:.4f}")
print(f"Average distance for DIFFERENT-species pairs (negative): {neg_mean:.4f}")
print(f"Suggested threshold (midpoint): {(pos_mean + neg_mean) / 2:.4f}")

NameError: name 'siamese_model' is not defined

In [3]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

THRESHOLD = 0.6766

siamese_model.eval()

all_true_labels = []
all_predictions = []

with torch.no_grad():
    for anchor, positive, negative in triplet_test_loader:
        anchor = anchor.to(processing_device)
        positive = positive.to(processing_device)
        negative = negative.to(processing_device)

        anchor_emb, positive_emb, negative_emb = siamese_model(anchor, positive, negative)

        pos_distance = torch.norm(anchor_emb - positive_emb, p=2, dim=1)
        neg_distance = torch.norm(anchor_emb - negative_emb, p=2, dim=1)

        all_true_labels.extend([1] * anchor.size(0))
        all_predictions.extend((pos_distance < THRESHOLD).int().cpu().numpy())

        all_true_labels.extend([0] * anchor.size(0))
        all_predictions.extend((neg_distance < THRESHOLD).int().cpu().numpy())

cm = confusion_matrix(all_true_labels, all_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted: Different", "Predicted: Same"],
            yticklabels=["True: Different", "True: Same"])
plt.title("Siamese Network — Same/Different Species Verification")
plt.tight_layout()
plt.savefig(r"D:\Camera_trap\reports\siamese_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

print(classification_report(all_true_labels, all_predictions, target_names=["Different species", "Same species"]))

NameError: name 'siamese_model' is not defined